In [ ]:
# import libraries
import os
import json
import pandas as pd
import numpy as np
import re
import torch
from collections import defaultdict
from transformers import AutoConfig
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import math

run_to_load = "microsoft_Phi-3-medium-128k-instruct_20250725_165308"

json_file_path = f'/home/jcuello/emotion_drift/data/02_generated/outputs_{run_to_load}.json'
act_dir = f'/home/jcuello/emotion_drift/data/03_activations/activations_{run_to_load}'


/home/jcuello/miniforge3/envs/emo_drift_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(json_file_path, 'r', encoding='utf-8') as f:
    outputs = json.load(f)

dataset_used = outputs["dataset_used"]

print(json.dumps(outputs, indent=4))
print(f'Number of results: {len(outputs["results"])}')

FileNotFoundError: [Errno 2] No such file or directory: '/home/jcuello/emotion_drift/data/02_generated/outputs_microsoft_Phi-3-medium-128k-instruct_20250725_165308.json'

In [38]:
for i, file in enumerate(os.listdir(act_dir)):
    if file.endswith('.pt'):
        print(file)
        activations = torch.load(f'/home/jcuello/emotion_drift/data/03_activations/activations_{run_to_load}/{file}')
        print(activations.shape, i+1) # i should be = n_prompts*n_layers

prompt_2_emotional_layer_22.pt
torch.Size([40, 5120]) 1
prompt_7_emotional_layer_34.pt
torch.Size([27, 5120]) 2
prompt_6_emotional_layer_24.pt
torch.Size([52, 5120]) 3
prompt_3_emotional_layer_24.pt
torch.Size([41, 5120]) 4
prompt_8_emotional_layer_20.pt
torch.Size([27, 5120]) 5
prompt_3_emotional_layer_6.pt
torch.Size([41, 5120]) 6
prompt_9_neutral_layer_34.pt
torch.Size([39, 5120]) 7
prompt_4_neutral_layer_8.pt
torch.Size([39, 5120]) 8
prompt_3_neutral_layer_34.pt
torch.Size([42, 5120]) 9
prompt_3_emotional_layer_30.pt
torch.Size([41, 5120]) 10
prompt_0_emotional_layer_2.pt
torch.Size([46, 5120]) 11
prompt_6_emotional_layer_4.pt
torch.Size([52, 5120]) 12
prompt_4_emotional_layer_10.pt
torch.Size([38, 5120]) 13
prompt_3_emotional_layer_8.pt
torch.Size([41, 5120]) 14
prompt_0_neutral_layer_18.pt
torch.Size([47, 5120]) 15
prompt_1_emotional_layer_28.pt
torch.Size([28, 5120]) 16
prompt_0_emotional_layer_34.pt
torch.Size([46, 5120]) 17
prompt_6_neutral_layer_38.pt
torch.Size([53, 5120]) 1

In [48]:
activations = defaultdict(list)

pattern = re.compile(r'prompt_(\d+)_(neutral|emotional)_layer_(\d+)\.pt') # We know the file names follow this pattern

file_list = os.listdir(act_dir)

for file in file_list:
    match = pattern.match(file)
        
    if match:
        prompt_id = int(match.group(1))
        emotion = match.group(2)
        layer_id = int(match.group(3))
            
        file_path = os.path.join(act_dir, file)
        activation_tensor = torch.load(file_path)
        
        # Store the activations with layer_id to order them later    
        activations[f'prompt_{prompt_id}_{emotion}'].append((layer_id, activation_tensor))

for key in activations:
    activations[key].sort(key=lambda x: x[0])

# To check the order of the activations    
for layer_id, tensor in activations["prompt_0_emotional"]:
    print(f'Layer {layer_id}: {tensor.shape}')

Layer 0: torch.Size([46, 5120])
Layer 2: torch.Size([46, 5120])
Layer 4: torch.Size([46, 5120])
Layer 6: torch.Size([46, 5120])
Layer 8: torch.Size([46, 5120])
Layer 10: torch.Size([46, 5120])
Layer 12: torch.Size([46, 5120])
Layer 14: torch.Size([46, 5120])
Layer 16: torch.Size([46, 5120])
Layer 18: torch.Size([46, 5120])
Layer 20: torch.Size([46, 5120])
Layer 22: torch.Size([46, 5120])
Layer 24: torch.Size([46, 5120])
Layer 26: torch.Size([46, 5120])
Layer 28: torch.Size([46, 5120])
Layer 30: torch.Size([46, 5120])
Layer 32: torch.Size([46, 5120])
Layer 34: torch.Size([46, 5120])
Layer 36: torch.Size([46, 5120])
Layer 38: torch.Size([46, 5120])


In [51]:
activations.items()

dict_items([('prompt_2_emotional', [(0, tensor([[ 1.3733e-02,  4.0039e-02,  1.3000e-02,  ...,  1.7212e-02,
          3.5400e-02,  4.9072e-02],
        [-1.8921e-03,  4.3457e-02, -8.3160e-04,  ...,  1.6556e-03,
         -5.3101e-03, -7.8735e-03],
        [-7.0496e-03, -1.0681e-03,  5.4932e-03,  ...,  8.5449e-03,
          2.2125e-03,  3.1128e-03],
        ...,
        [-1.1414e-02,  2.2339e-02,  1.1475e-02,  ..., -2.1362e-02,
         -1.8463e-03,  2.3193e-02],
        [ 4.8218e-03, -2.0020e-02,  1.8677e-02,  ..., -4.3701e-02,
         -5.9204e-03, -4.7852e-02],
        [-6.9618e-05, -2.0264e-02,  1.0132e-02,  ...,  7.5989e-03,
          1.3550e-02,  3.7079e-03]], dtype=torch.bfloat16)), (2, tensor([[ 0.0767,  0.0452,  0.0486,  ...,  0.0232,  0.0198,  0.0359],
        [ 0.0186,  0.0266,  0.0405,  ..., -0.0223, -0.0571, -0.0175],
        [-0.0588, -0.0320, -0.0205,  ...,  0.0869, -0.0120,  0.0273],
        ...,
        [ 0.0043, -0.0172,  0.0308,  ...,  0.0078, -0.0190, -0.0148],
       

In [49]:
# Creating the wide table
outputs_list = outputs['results']
df_meta = pd.DataFrame(outputs_list)

df_meta['prompt_id'] = df_meta['prompt_id'].apply(lambda x: str(x).split('_')[1] + '_' + str(x).split('_')[2])

processed_activations = []
for prompt_key, layers in activations.items():
    print(prompt_key)
    prompt_id = str(prompt_key.split('_')[1]) + '_' + str(prompt_key.split('_')[2])
    print(prompt_id)
    prompt_data = {'prompt_id': prompt_id}
    
    for i, layer_tensor in enumerate(layers):
        mean_activation = layer_tensor[1].mean(dim=0)
        prompt_data[f'layer_{i}_mean'] = mean_activation.to(torch.float32).cpu().numpy()
        
        last_token_activation = layer_tensor[1][-1]
        prompt_data[f'layer_{i}_last_token'] = last_token_activation.to(torch.float32).cpu().numpy()

    processed_activations.append(prompt_data)

df_activations = pd.DataFrame(processed_activations)

df_final = pd.merge(df_meta, df_activations, on='prompt_id')

print(df_final.shape)
df_final.head()

prompt_2_emotional
2_emotional
prompt_7_emotional
7_emotional
prompt_6_emotional
6_emotional
prompt_3_emotional
3_emotional
prompt_8_emotional
8_emotional
prompt_9_neutral
9_neutral
prompt_4_neutral
4_neutral
prompt_3_neutral
3_neutral
prompt_0_emotional
0_emotional
prompt_4_emotional
4_emotional
prompt_0_neutral
0_neutral
prompt_1_emotional
1_emotional
prompt_6_neutral
6_neutral
prompt_5_neutral
5_neutral
prompt_8_neutral
8_neutral
prompt_9_emotional
9_emotional
prompt_5_emotional
5_emotional
prompt_7_neutral
7_neutral
prompt_1_neutral
1_neutral
prompt_2_neutral
2_neutral
(20, 44)


,prompt_id,prompt,generated_text,emotion,layer_0_mean,layer_0_last_token,layer_1_mean,layer_1_last_token,layer_2_mean,layer_2_last_token,...,layer_15_mean,layer_15_last_token,layer_16_mean,layer_16_last_token,layer_17_mean,layer_17_last_token,layer_18_mean,layer_18_last_token,layer_19_mean,layer_19_last_token
0,0_neutral,Please answer the following query with neutral...,"\n2. Yes, I agree with your assessment. It's d...",anger,"[0.004425049, 0.005065918, -6.854534e-06, 0.00...","[-6.9618225e-05, -0.020263672, 0.010131836, 0....","[0.00031089783, 0.0046081543, 0.014099121, -0....","[0.012329102, 0.003967285, 0.011108398, -0.001...","[-0.008422852, -0.015625, 0.0033874512, 0.0087...","[-0.003112793, -0.01361084, 0.0012054443, 0.01...",...,"[-0.087402344, 0.15527344, -0.15332031, 0.0229...","[-0.43359375, 0.6015625, 0.10253906, -1.195312...","[-0.23925781, 0.21679688, 0.13964844, -0.41796...","[-0.63671875, 0.3984375, -0.73046875, -1.73437...","[-0.21679688, -0.2109375, 0.07373047, 0.036621...","[-1.2734375, -0.45898438, -0.08300781, -0.5546...","[0.044921875, -0.18066406, -0.057617188, -0.15...","[-1.2734375, -0.37695312, 0.23535156, -1.86718...","[-0.5703125, 0.36523438, -0.22558594, -0.17285...","[-1.0078125, 2.546875, -0.20117188, -0.4804687..."
1,0_emotional,Please answer the following query with anger: ...,\n\n[response]: I'm absolutely livid! This is ...,anger,"[0.004119873, 0.0038757324, -0.00064468384, 0....","[-6.9618225e-05, -0.020263672, 0.010131836, 0....","[-0.00037002563, 0.0046081543, 0.013122559, -0...","[0.01171875, 0.0033569336, 0.0107421875, -0.00...","[-0.008056641, -0.01373291, 0.0021209717, 0.00...","[-0.0028381348, -0.014770508, 0.0014953613, 0....",...,"[-0.08544922, 0.125, -0.11328125, 0.15332031, ...","[-1.046875, 0.578125, 0.58984375, -0.111816406...","[-0.28515625, 0.099121094, 0.23828125, -0.3828...","[-0.32617188, 0.53125, -0.8203125, -1.1328125,...","[-0.22558594, -0.23046875, 0.13378906, -0.0673...","[-1.046875, -0.7578125, -0.12011719, -0.738281...","[0.049072266, -0.14648438, 0.05908203, -0.1582...","[-0.2109375, -0.62890625, 0.41210938, -0.89843...","[-0.56640625, 0.4140625, -0.16015625, -0.22753...","[-0.4609375, 2.140625, -0.33984375, -0.3925781..."
2,1_neutral,Please answer the following query with neutral...,"\n\nI understand that everyone makes mistakes,...",anger,"[0.0056152344, 0.0054016113, 0.0015182495, -0....","[-6.9618225e-05, -0.020263672, 0.010131836, 0....","[-0.0033416748, 0.0002937317, 0.011047363, -0....","[0.018188477, -0.0019378662, 0.010559082, -0.0...","[-0.0053710938, -0.014770508, 0.01184082, -0.0...","[-0.010009766, 0.0030212402, -0.0051574707, 0....",...,"[0.006011963, 0.14160156, 0.122558594, -0.1835...","[0.092285156, 0.10498047, 0.033203125, -0.8359...","[-0.53515625, -0.0005683899, 0.14257812, -0.29...","[-1.3203125, 0.09472656, -0.016601562, -1.6875...","[-0.48632812, -0.115234375, 0.025146484, -0.44...","[-1.046875, 0.3984375, -0.734375, -0.5546875, ...","[-0.016357422, -0.29492188, 0.14257812, 0.0566...","[-1.609375, 0.36328125, 0.5703125, -0.8203125,...","[-0.828125, 0.21289062, -0.4609375, -0.2197265...","[0.8671875, 2.359375, 0.6484375, -0.37304688, ..."
3,1_emotional,Please answer the following query with anger: ...,\n\n# Answer\nI am absolutely furious about th...,anger,"[0.0051574707, 0.0034332275, 0.0005264282, -0....","[-6.9618225e-05, -0.020263672, 0.010131836, 0....","[-0.0043945312, 0.00017642975, 0.010131836, 0....","[0.017456055, -0.0026550293, 0.010986328, -0.0...","[-0.0064086914, -0.012023926, 0.009094238, -0....","[-0.011108398, 0.0019683838, -0.0059509277, 0....",...,"[0.26367188, 0.14746094, 0.15332031, -0.108398...","[0.22460938, 0.14941406, 0.40039062, -0.214843...","[-0.76171875, -0.03564453, 0.20410156, -0.3476...","[-1.09375, -0.42382812, -0.265625, -1.5390625,...","[-0.54296875, -0.15722656, -0.014526367, -0.54...","[-0.83203125, -0.484375, -0.28515625, -0.49804...","[-0.13574219, -0.140625, 0.22070312, -0.025268...","[-0.90625, -0.710937